# ATML PA1 -- Colab bootstrap

Run this once per session. It mounts Drive (for datasets/checkpoints, which are git-ignored
and must survive a disconnect), clones/pulls this repo (the source of truth for code and
directory structure), and installs dependencies. After this, run everything as
`!python -m task2.train --config task2/configs/dann.yaml`-style commands from the repo root
so the exact same scripts work locally too.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/atml_pa1'
DATA_ROOT = f'{DRIVE_ROOT}/data'       # STL-10 / PACS / CIFAR-10 / CIFAR-100 cache
CKPT_ROOT = f'{DRIVE_ROOT}/checkpoints' # everything task*/results/*/checkpoint.pt points at

import os
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)
print('Data root:', DATA_ROOT)
print('Checkpoint root:', CKPT_ROOT)

In [ ]:
REPO_URL = 'https://github.com/adilawan1/ATML-PA1.git'
REPO_DIR = '/content/ATML-PA1'

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd $REPO_DIR

## Git identity + GitHub auth (run once per session, before any commit cell)

Colab starts with neither a git identity nor GitHub push credentials, and neither survives a
runtime reset -- this needs to run every fresh session, not just once ever.

1. If you don't already have one, create a GitHub Personal Access Token (classic, `repo`
   scope) at github.com/settings/tokens.
2. In this notebook's left sidebar, click the key icon ("Secrets"), add a new secret named
   `GH_TOKEN` with that token as the value, and enable notebook access for it.
3. Run the cell below (edit the placeholder name on its first line first).

In [ ]:
# EDIT the name below to your real name first -- your local machine's git config has
# user.name set to a stray "=", don't copy that here.
!git config --global user.name "YOUR NAME HERE"
!git config --global user.email "1ahmed2adil3awan@gmail.com"

from google.colab import userdata
GH_TOKEN = userdata.get('GH_TOKEN')
!git remote set-url origin https://{GH_TOKEN}@github.com/adilawan1/ATML-PA1.git
print("git identity + auth configured for this session")

In [ ]:
# IMPORTANT: do NOT `pip install torch`/`torchvision` here. Colab preinstalls a build of
# each already matched to its GPU driver; PyPI's default (untagged) wheel for both is
# CPU-only, and reinstalling them is exactly what causes
# "AssertionError: Torch not compiled with CUDA enabled" later on. Install everything else
# from requirements.txt and leave those two alone.
!grep -vE '^(torch|torchvision)$' requirements.txt > /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt

# Sanity check -- if this ever prints False/None on a GPU runtime, something (re)installed a
# CPU-only torch; fix with:
#   !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
import torch
print("torch", torch.__version__, "| cuda build:", torch.version.cuda, "| available:", torch.cuda.is_available())

In [ ]:
!nvidia-smi

## Task 4 kickoff (Vanilla + GCSC)

These are the two required Task 4 trainings that need no manual dataset download (CIFAR-10
fetches automatically) and don't depend on PACS. Each cell checks for an existing checkpoint
first, so it's safe to re-run this section if the runtime disconnects mid-way -- it will skip
anything already finished rather than retraining from scratch. Run the three cells below in
order, then the commit cell once both are done.

In [ ]:
import os

SPLIT_PATH = "task4/data/cifar10_split_seed6304.json"
if not os.path.exists(SPLIT_PATH):
    !python -m task4.data.make_splits --data-root {DATA_ROOT}
else:
    print(f"{SPLIT_PATH} already exists, skipping.")

In [ ]:
import os

from task4.methods.vanilla import train_vanilla

VANILLA_CKPT = f"{CKPT_ROOT}/task4/vanilla/checkpoint.pt"
if os.path.exists(VANILLA_CKPT):
    print(f"Found existing checkpoint at {VANILLA_CKPT}, skipping training. Delete it to retrain.")
else:
    vanilla_model = train_vanilla(
        data_root=DATA_ROOT,
        split_path="task4/data/cifar10_split_seed6304.json",
        device="cuda",
        checkpoint_path=VANILLA_CKPT,
        metrics_path="task4/results/vanilla/metrics.jsonl",
    )

In [ ]:
import os

from task4.methods.gcsc import train_gcsc

GCSC_CKPT = f"{CKPT_ROOT}/task4/gcsc/checkpoint.pt"
if os.path.exists(GCSC_CKPT):
    print(f"Found existing checkpoint at {GCSC_CKPT}, skipping training. Delete it to retrain.")
else:
    gcsc_model = train_gcsc(
        data_root=DATA_ROOT,
        split_path="task4/data/cifar10_split_seed6304.json",
        device="cuda",
        checkpoint_path=GCSC_CKPT,
        metrics_path="task4/results/gcsc/metrics.jsonl",
    )

In [ ]:
!git pull
!git add task4/data/cifar10_split_seed6304.json task4/results/vanilla/metrics.jsonl task4/results/gcsc/metrics.jsonl
!git commit -m "Add Task 4 Vanilla and GCSC training curves"
!git push

## Task 4 evaluation (run once Vanilla + GCSC checkpoints exist)

Caches each checkpoint's features/logits, then builds both required tables (post-hoc scores
on Vanilla; Vanilla/GCSC comparison via MLS), the score-distribution figure, and the
near/far failure cases. Safe to re-run -- `extract_outputs` overwrites its own cache files,
and `evaluate_osr` just recomputes tables from whatever caches currently exist (it'll pick up
a PROSER row automatically once that's trained and cached the same way).

In [ ]:
from task4.extract_outputs import extract_all_outputs

extract_all_outputs(
    checkpoint_path=VANILLA_CKPT,
    data_root=DATA_ROOT,
    method_name="vanilla",
    device="cuda",
)
extract_all_outputs(
    checkpoint_path=GCSC_CKPT,
    data_root=DATA_ROOT,
    method_name="gcsc",
    device="cuda",
)

In [ ]:
from task4.evaluate_osr import main as evaluate_osr_main

evaluate_osr_main(data_root=DATA_ROOT, cache_dir="task4/cache")

In [ ]:
!git pull
!git add task4/results/table1_vanilla_posthoc_scores.json task4/results/table2_model_comparison_mls.json task4/results/failure_cases.json report/figures/task4_score_distributions.png
!git commit -m "Add Task 4 Vanilla/GCSC evaluation tables, figure, and failure cases"
!git push

## PACS dataset (needed for Tasks 2 and 3)

Download -> copy into Drive -> verify the layout/image counts (Photo 1670, Art 2048,
Cartoon 2344, Sketch 3929) -> build and commit the shared split protocol. Skips whatever is
already done, so it's safe to re-run.

In [ ]:
import os

PACS_DIR = f"{DATA_ROOT}/pacs"
if os.path.isdir(f"{PACS_DIR}/photo"):
    print("PACS already on Drive at", PACS_DIR, "-- skipping download.")
else:
    !pip install -q gdown
    # Same Drive file DomainBed's download script uses. If Drive reports a download-quota
    # error, retry later or fall back to the Kaggle mirror (nickfratto/pacs-dataset).
    !gdown "https://drive.google.com/uc?id=1JFr8f805nMUelQWWmfnJR3y4_SYoN5Pd" -O /content/pacs.zip
    !unzip -q -o /content/pacs.zip -d /content/pacs_extracted
    !find /content/pacs_extracted -maxdepth 2 | head -20

In [ ]:
import glob
import os
import shutil

PACS_DIR = f"{DATA_ROOT}/pacs"
if os.path.isdir(f"{PACS_DIR}/photo"):
    print("PACS already on Drive at", PACS_DIR)
else:
    # The archive's top folder name varies by mirror (DomainBed's is "kfold"); find whichever
    # folder directly contains photo/ and copy it into Drive. Copying ~10k small files to Drive
    # takes a few minutes.
    hits = [os.path.dirname(p) for p in glob.glob("/content/pacs_extracted/**/photo", recursive=True) if os.path.isdir(p)]
    assert hits, "No 'photo' folder found in /content/pacs_extracted -- inspect the archive layout"
    print("Copying", hits[0], "->", PACS_DIR)
    shutil.copytree(hits[0], PACS_DIR)

In [ ]:
from shared.verify_pacs import verify

PACS_DIR = f"{DATA_ROOT}/pacs"
assert verify(PACS_DIR), "Fix the PACS layout above before building the split protocol"
!python -m shared.pacs_protocol --root {PACS_DIR}

!git pull
!git add shared/splits/pacs_sketch_seed6304.json
!git commit -m "Add PACS split protocol (seed 6304; Photo/Art/Cartoon source, Sketch target)"
!git push

## Task 1 -- clean baseline, color, translation, patch shuffle, representation analysis

Before running: write your hypotheses in `task1/hypotheses.md` and commit them -- the
assignment wants each design choice's hypothesis stated before its result is interpreted.
STL-10 downloads (~2.6 GB) into `DATA_ROOT` on Drive the first time; frozen train/val
features are cached on Drive so re-runs skip extraction. Cue conflicts (Step 3) are a
separate script, added below once AdaIN is wired in.

In [ ]:
!python -m task1.scripts.run_task1 --data-root {DATA_ROOT} --cache-dir {DRIVE_ROOT}/cache/task1

In [ ]:
!git pull
!git add task1/data/eval_subset_seed6304.json task1/results/task1_results.json task1/results/compact_comparison.csv report/figures/task1_translation_curve.png report/figures/task1_tsne.png
!git commit -m "Add Task 1 results: clean baseline, color, translation, patch shuffle, representation stability"
!git push

## Running a task

Point every `--data-root` / `pacs_root` at `DATA_ROOT`, and every config's `output.checkpoint`
at a path under `CKPT_ROOT` (edit the YAML, or symlink `task2/results` etc. into
`CKPT_ROOT` -- either works, just be consistent so a disconnect doesn't lose a checkpoint).

```python
!python -m shared.pacs_protocol --root {DATA_ROOT}/pacs
!python -m task2.train --config task2/configs/source_only.yaml
```

After any run that produced new small result files (JSON/CSV/figures, not checkpoints),
commit and push from a cell:

```python
!git add task2/results/*.json report/figures
!git commit -m "Add Task 2 source-only results"
!git push
```